In [2]:
master="spark://spark-master:7077"

SETUP DO SPARK DO LAB3 ABAIXO, TALVEZ PRECISA DE MAIS DEFINIÇOES

In [ ]:
from spark_profiler import SparkProfiler
from pyspark.sql import SparkSession


spark_builder = (SparkSession.builder
         .master(master)
         .appName("spark-profiling-showcase")
         .config("spark.ui.showConsoleProgress", "false"))

spark = spark_builder.getOrCreate()

print("Spark UI:", spark.sparkContext.uiWebUrl)
print("Default parallelism (cores seen by the driver):", spark.sparkContext.defaultParallelism)

In [3]:
import glob
from functools import reduce
from pyspark.sql import SparkSession
from pyspark.sql import functions as sf
from pyspark.sql.functions import col, to_timestamp

spark = SparkSession.builder \
             .master(master) \
             .appName('taxi_zone') \
             .getOrCreate()


zones = (spark.read.option("delimiter", ",").option("header", True)
         .csv("/data/taxi_zone_lookup.csv"))
trips = spark.read.parquet("/data/yellow_tripdata_2023-**.parquet")

def organize(df):
    return df.select(
    # Converte as datas de STRING para TIMESTAMP de forma segura
    to_timestamp(col("tpep_pickup_datetime")).alias("tpep_pickup_datetime"),
    to_timestamp(col("tpep_dropoff_datetime")).alias("tpep_dropoff_datetime"),
    col("PULocationID").cast("int").alias("PULocationID"),
    col("DOLocationID").cast("int").alias("DOLocationID"),
    col("passenger_count").cast("int"),
    col("trip_distance").cast("double"),
    col("fare_amount").cast("double"),
    col("tip_amount").cast("double"),
    col("payment_type").cast("double"),
    col("total_amount").cast("double")
)

files = sorted(glob.glob("/data/yellow_tripdata_2023-*.parquet"))
df_organized = reduce(lambda a, b: a.unionByName(b),
                      [organize(spark.read.parquet(f)) for f in files])
# one copy of the lookup per join, with its columns renamed so the two don't clash
pu = zones.select(sf.col("LocationID").cast("int").alias("PULocationID"),
                  sf.col("Borough").alias("PU_Borough"),
                  sf.col("Zone").alias("PU_Zone"),
                  sf.col("service_zone").alias("PU_service_zone"))
do = zones.select(sf.col("LocationID").cast("int").alias("DOLocationID"),
                  sf.col("Borough").alias("DO_Borough"),
                  sf.col("Zone").alias("DO_Zone"),
                  sf.col("service_zone").alias("DO_service_zone"))


joined = (df_organized.join(pu, on="PULocationID", how="left")
                      .join(do, on="DOLocationID", how="left"))
joined.show()


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/10/26 01:26:02 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/10/26 01:26:10 WARN FileStreamSink: Assume no metadata directory. Error while looking for metadata directory in the path: /data/yellow_tripdata_2023-**.parquet.
java.io.FileNotFoundException: File /data/yellow_tripdata_2023-**.parquet does not exist
	at org.apache.hadoop.fs.RawLocalFileSystem.deprecatedGetFileStatus(RawLocalFileSystem.java:980)
	at org.apache.hadoop.fs.RawLocalFileSystem.getFileLinkStatusInternal(RawLocalFileSystem.java:1301)
	at org.apache.hadoop.fs.RawLocalFileSystem.getFileStatus(RawLocalFileSystem.java:970)
	at org.apache.hadoop.fs.FilterFileSystem.getFileStatus(FilterFileSystem.java:462)
	at org.apache.spark.sql.e

+------------+------------+--------------------+---------------------+---------------+-------------+-----------+----------+------------+------------+----------+--------------------+---------------+----------+--------------------+---------------+
|DOLocationID|PULocationID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|fare_amount|tip_amount|payment_type|total_amount|PU_Borough|             PU_Zone|PU_service_zone|DO_Borough|             DO_Zone|DO_service_zone|
+------------+------------+--------------------+---------------------+---------------+-------------+-----------+----------+------------+------------+----------+--------------------+---------------+----------+--------------------+---------------+
|         141|         161| 2023-01-01 00:32:10|  2023-01-01 00:40:36|              1|         0.97|        9.3|       0.0|         2.0|        14.3| Manhattan|      Midtown Center|    Yellow Zone| Manhattan|     Lenox Hill West|    Yellow Zone|
|         237|  



M1.


In [4]:

result1 = df_organized.where("""
    fare_amount> 0
    AND tpep_dropoff_datetime IS NOT NULL
    AND trip_distance > 0
    AND passenger_count BETWEEN 1 AND 6
    AND tpep_dropoff_datetime > tpep_pickup_datetime
    AND PULocationID BETWEEN 1 AND 263
    AND DOLocationID BETWEEN 1 AND 263
""")

result1.show()



+--------------------+---------------------+------------+------------+---------------+-------------+-----------+----------+------------+------------+
|tpep_pickup_datetime|tpep_dropoff_datetime|PULocationID|DOLocationID|passenger_count|trip_distance|fare_amount|tip_amount|payment_type|total_amount|
+--------------------+---------------------+------------+------------+---------------+-------------+-----------+----------+------------+------------+
| 2023-01-01 00:32:10|  2023-01-01 00:40:36|         161|         141|              1|         0.97|        9.3|       0.0|         2.0|        14.3|
| 2023-01-01 00:55:08|  2023-01-01 01:01:27|          43|         237|              1|          1.1|        7.9|       4.0|         1.0|        16.9|
| 2023-01-01 00:25:04|  2023-01-01 00:37:49|          48|         238|              1|         2.51|       14.9|      15.0|         1.0|        34.9|
| 2023-01-01 00:10:29|  2023-01-01 00:21:19|         107|          79|              1|         1.43|

M2.


In [7]:

result2 = (joined
    .withColumn("day_of_week", sf.dayofweek("tpep_pickup_datetime"))
    .withColumn("hour_of_day", sf.hour("tpep_pickup_datetime"))
    .withColumn("duration_min",
                (sf.col("tpep_dropoff_datetime").cast("long")
                 - sf.col("tpep_pickup_datetime").cast("long")) / 60)
    .groupBy(sf.col("PU_Borough").alias("pickup_borough"), "day_of_week", "hour_of_day")
    .agg(sf.count("*").alias("trip_count"),
         sf.avg("fare_amount").alias("mean_fare_amount"),
         sf.avg("duration_min").alias("mean_duration_min"),
         sf.avg("trip_distance").alias("mean_trip_distance")))

(result2.write
      .mode("overwrite")
      .partitionBy("pickup_borough")
      .parquet("/data/output/borough_hour_stats"))

In [6]:
spark.read.parquet("/data/output/borough_hour_stats").show()

+-----------+-----------+----------+------------------+------------------+------------------+--------------+
|day_of_week|hour_of_day|trip_count|  mean_fare_amount| mean_duration_min|mean_trip_distance|pickup_borough|
+-----------+-----------+----------+------------------+------------------+------------------+--------------+
|          1|         15|    255042|16.989604535723515|16.349904852272697| 3.578299221304725|     Manhattan|
|          2|         16|    270817|  15.7922868579151|15.944021547145597|2.6178271304977145|     Manhattan|
|          3|         11|    256509| 16.40734847510223|16.913721870707604| 2.279293553052723|     Manhattan|
|          6|         21|    272052|14.873464447973179|  14.6445650708933|2.4662505329863382|     Manhattan|
|          3|          7|    171535|14.531897863409823|12.351070238338139| 4.035595592736176|     Manhattan|
|          6|          2|     51920|15.264525231124814|12.691661530559836|3.4563819337442228|     Manhattan|
|          2|      

M3.


In [23]:
from pyspark.sql import Window

w_rank = Window.partitionBy("pickup_borough", "month_idx").orderBy(sf.desc("trips"))
w_roll = Window.partitionBy("pickup_borough", "PULocationID").orderBy("month_idx").rangeBetween(-2, 0)
w_bor  = Window.partitionBy("pickup_borough", "month_idx")

result3 = (df_organized
    .join(pu, on="PULocationID", how="left")
    .withColumn("month", sf.date_trunc("month", "tpep_pickup_datetime"))
    .withColumn("month_idx", sf.year("month") * 12 + sf.month("month"))
    .groupBy(sf.col("PU_Borough").alias("pickup_borough"),
             "PULocationID", "PU_Zone", "month_idx", "month")
    .agg(sf.count("*").alias("trips"))
    .withColumn("rank_in_borough", sf.rank().over(w_rank))
    .withColumn("roll3_trips", sf.sum("trips").over(w_roll))
    .withColumn("roll3_share", sf.col("roll3_trips") / sf.sum("roll3_trips").over(w_bor))
    .withColumn("roll_mean", sf.avg("trips").over(w_roll))
    .withColumn("roll_std", sf.stddev("trips").over(w_roll))
    .withColumn("anomaly",
                sf.coalesce(sf.abs(sf.col("trips") - sf.col("roll_mean")) > 1.5 * sf.col("roll_std"),
                            sf.lit(False))))

result3.show()

+--------------+------------+--------------------+---------+-------------------+-----+---------------+-----------+--------------------+------------------+------------------+-------+
|pickup_borough|PULocationID|             PU_Zone|month_idx|              month|trips|rank_in_borough|roll3_trips|         roll3_share|         roll_mean|          roll_std|anomaly|
+--------------+------------+--------------------+---------+-------------------+-----+---------------+-----------+--------------------+------------------+------------------+-------+
|         Bronx|           3|Allerton/Pelham G...|    24277|2023-01-01 00:00:00|   39|             35|         39| 0.00937049495434887|              39.0|              NULL|  false|
|         Bronx|           3|Allerton/Pelham G...|    24278|2023-02-01 00:00:00|   45|             32|         84| 0.01006108515989939|              42.0| 4.242640687119285|  false|
|         Bronx|           3|Allerton/Pelham G...|    24279|2023-03-01 00:00:00|   61|    

O1

In [25]:
result4 = (df_organized
    .filter(sf.col("trip_distance") > 0)
    .join(pu, on="PULocationID", how="left")
    .withColumn("hour_of_day", sf.hour("tpep_pickup_datetime"))
    .withColumn("mean_fare_amount",
                sf.avg(df_organized["fare_amount"]/df_organized["trip_distance"])
                  .over(Window.partitionBy("PULocationID", "hour_of_day")))
    .withColumn("sigma_fare_amount",
                sf.stddev(df_organized["fare_amount"]/df_organized["trip_distance"])
                  .over(Window.partitionBy("PULocationID", "hour_of_day")))
    .withColumn("anomaly",
                sf.coalesce((df_organized["fare_amount"]/df_organized["trip_distance"])
                            > sf.col("mean_fare_amount") + 3 * sf.col("sigma_fare_amount"),
                            sf.lit(False)))
    .withColumn("day_type",
                sf.when(sf.dayofweek("tpep_pickup_datetime").isin(1, 7), "weekend").otherwise("weekday"))
    .groupBy(sf.col("PU_Borough").alias("pickup_borough"), "payment_type", "day_type")
    .agg(sf.count("*").alias("trips"),
         sf.sum(sf.col("anomaly").cast("int")).alias("anomalies"))
    .withColumn("anomaly_rate", sf.col("anomalies") / sf.col("trips"))
    .groupBy("pickup_borough", "payment_type")
    .pivot("day_type", ["weekday", "weekend"])
    .agg(sf.first("anomaly_rate"))
    .withColumn("weekend_minus_weekday", sf.col("weekend") - sf.col("weekday")))

result4.orderBy("pickup_borough", "payment_type").show(50)

+--------------+------------+--------------------+--------------------+---------------------+
|pickup_borough|payment_type|             weekday|             weekend|weekend_minus_weekday|
+--------------+------------+--------------------+--------------------+---------------------+
|         Bronx|         0.0|0.008175399479747306|0.003956478733926805| -0.00421892074582...|
|         Bronx|         1.0|0.019235146824493513|0.023095077545515846| 0.003859930721022333|
|         Bronx|         2.0|0.015007724564113882|0.012542209358417752| -0.00246551520569613|
|         Bronx|         3.0| 0.04119850187265917| 0.06862745098039216|  0.02742894910773299|
|         Bronx|         4.0|0.030848329048843187| 0.05851063829787234| 0.027662309249029154|
|      Brooklyn|         0.0|0.004140616430843...|0.003816493547371...| -3.24122883471745E-4|
|      Brooklyn|         1.0|0.011361990906545936|0.007135644564119825| -0.00422634634242...|
|      Brooklyn|         2.0|0.009813906119803721|0.00943632

O3